# TCGA-BRCA Treatment Groups Descriptive Review V1

This notebook is review-only. It reads the latest saved treatment-groups descriptive-review outputs from disk,
regenerates review tables in `05-results`, and does not reread raw treatment tables, normalize drug names,
freeze treatment arms, perform modeling, or make treatment recommendations.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_treatment_groups_descriptive_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest treatment-groups descriptive v1 pointer not found: {latest_pointer_path}. '
        'Run script 22 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
group_summary_path = repo_root / latest_pointer['treatment_groups_descriptive_v1_group_summary_tsv']
confounder_coverage_path = repo_root / latest_pointer['treatment_groups_descriptive_v1_confounder_coverage_tsv']
value_composition_path = repo_root / latest_pointer['treatment_groups_descriptive_v1_value_composition_tsv']
manual_review_summary_path = repo_root / latest_pointer['treatment_groups_descriptive_v1_manual_review_summary_tsv']
summary_path = repo_root / latest_pointer['treatment_groups_descriptive_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [
    group_summary_path,
    confounder_coverage_path,
    value_composition_path,
    manual_review_summary_path,
    summary_path,
    run_log_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required descriptive-review artifact not found: {required_path}')

group_summary_df = read_tsv(group_summary_path)
confounder_coverage_df = read_tsv(confounder_coverage_path)
value_composition_df = read_tsv(value_composition_path)
manual_review_summary_df = read_tsv(manual_review_summary_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError('run_log.json does not report status == completed.')
if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if group_summary_df.empty:
    raise ValueError('treatment_groups_descriptive_v1_group_summary.tsv contains no rows.')

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

print(f"Run ID            : {latest_pointer['treatment_groups_descriptive_v1_run_id']}")
print(f"Grouping run ID   : {latest_pointer['patient_treatment_grouping_v1_run_id']}")
print(f"OS endpoint run ID: {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Pointer           : {latest_pointer_path}")


In [ ]:
review_group_summary_path = results_root / '121_treatment_groups_descriptive_v1_group_summary.tsv'
review_confounder_coverage_path = results_root / '122_treatment_groups_descriptive_v1_confounder_coverage.tsv'
review_value_composition_path = results_root / '123_treatment_groups_descriptive_v1_value_composition.tsv'
review_manual_review_summary_path = results_root / '124_treatment_groups_descriptive_v1_manual_review_summary.tsv'
review_summary_path = results_root / '125_treatment_groups_descriptive_v1_summary.tsv'

group_summary_df.to_csv(review_group_summary_path, sep='\t', index=False)
confounder_coverage_df.to_csv(review_confounder_coverage_path, sep='\t', index=False)
value_composition_df.to_csv(review_value_composition_path, sep='\t', index=False)
manual_review_summary_df.to_csv(review_manual_review_summary_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_group_summary_path}')
print(f'Saved: {review_confounder_coverage_path}')
print(f'Saved: {review_value_composition_path}')
print(f'Saved: {review_manual_review_summary_path}')
print(f'Saved: {review_summary_path}')


In [ ]:
print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation_df = pd.DataFrame(
    [{'check': key, 'value': str(value)} for key, value in run_log.get('validation', {}).items()]
)
print('\n=== Validation ===')
display(validation_df)

counts_df = pd.DataFrame(
    [{'metric': key, 'value': str(value)} for key, value in run_log.get('counts', {}).items()]
)
print('\n=== Key counts ===')
display(counts_df)

print('\n=== Summary TSV ===')
display(summary_df)


In [ ]:
print('=== Group summary ===')
display(group_summary_df)

print('\n=== Manual-review summary ===')
display(manual_review_summary_df)

coverage_focus_df = confounder_coverage_df[
    confounder_coverage_df['field_name'].isin(
        ['er_status_by_ihc', 'pr_status_by_ihc', 'her2_status_by_ihc', 'ajcc_pathologic_tumor_stage']
    )
].reset_index(drop=True)
print('\n=== Coverage focus ===')
display(coverage_focus_df)

age_composition_df = value_composition_df[
    value_composition_df['field_name'] == 'age_at_diagnosis'
].reset_index(drop=True)
print('\n=== Age summaries by group ===')
display(age_composition_df)
